# Circuit 2 — GHZ State (15 qubits)

**What it does:** Prepares a Greenberger–Horne–Zeilinger (GHZ) state across 15 qubits.
A single Hadamard on q0 creates a superposition, then a cascade of CNOTs spreads
entanglement to q1 through q14, producing `(|000...0⟩ + |111...1⟩) / √2`.

**Notable:** GHZ is all-Clifford (H + CNOT), so `ManyShotRunner` uses the same circuit
as the purity pass — no stand-in needed. Errors propagate dramatically across the
chain: a single CNOT error can flip the parity of every subsequent qubit.

**Purity pass memory note:** The 15-qubit density matrix is ~8 GB. The code tries
15q first and automatically falls back to a 13-qubit circuit (~512 MB) if an
out-of-memory error is raised. The heatmap and GIF always run at 15 qubits.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import noisiq as nq
from noisiq.backends import TrajectoryBackend, ManyShotRunner
from noisiq.noise import fill_idle_with_identities
from noisiq.visualization import (
    Visualizer,
    export_gif,
    interactive_heatmap,
    plot_error_heatmap,
    per_qubit_purities,
    add_purity_panel,
)

os.makedirs("outputs", exist_ok=True)

profile    = nq.noise.get_hardware("ibm_eagle_r3")
gate_times = profile.gate_times

N_SHOTS      = 2000
N_SHOTS_TRAJ = 400

print(f"noisiq {nq.__version__}")
print(profile.describe())

In [ ]:
def run_purity_pass(circuit, noise_config_twirl, n_qubits, label=""):
    result_traj = TrajectoryBackend().run(
        circuit, noise_model=noise_config_twirl, n_shots=N_SHOTS_TRAJ, seed=42,
    )
    rho = result_traj.final_state
    purities = per_qubit_purities(rho, n_qubits)
    print(f"\n{'─'*50}")
    print(f"Per-qubit purity  [{label}]")
    for q, p in enumerate(purities):
        print(f"  q{q:>2d}  Tr(ρ²) = {p:.4f}  {'█' * int(p * 20)}")
    print(f"{'─'*50}\n")
    return rho, purities


def visualize_circuit(circuit, result_many, noise_pauli, rho, purities,
                      label, gif_name, n_qubits_purity=None):
    """n_qubits_purity overrides circuit.n_qubits for the purity panel (OOM fallback)."""
    n_purity = n_qubits_purity if n_qubits_purity is not None else circuit.n_qubits

    fig = plot_error_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        title=f"IBM Eagle r3 · {label}",
    )
    add_purity_panel(fig, fig.axes[0], rho, n_purity)
    plt.show()

    interactive_heatmap(
        result_many, circuit, noise_config=noise_pauli,
        display_mode="annotate",
        title=f"IBM Eagle r3 · {label} [interactive]",
    )
    plt.show()

    viz = Visualizer(circuit)
    viz.many_shot_result = result_many
    gif_path = f"outputs/{gif_name}.gif"
    export_gif(viz, gif_path, purities=purities)
    print(f"GIF → {gif_path}")


print("Helpers ready.")

In [ ]:
N_GHZ = 15

def build_ghz_circuit(n: int) -> nq.Circuit:
    """Standard GHZ prep: H on q0, cascaded CNOTs q0→q1→…→q_{n-1}."""
    c = nq.Circuit(n_qubits=n, name=f"ghz_{n}q")
    c.h(0, t=0)
    for i in range(n - 1):
        c.cnot(i, i + 1, t=i + 1)
    return c


circuit_ghz = build_ghz_circuit(N_GHZ)
circuit_ghz_filled = fill_idle_with_identities(circuit_ghz, gate_times)

print(f"GHZ circuit ({N_GHZ}q) ops:        {len(circuit_ghz.operations)}")
print(f"GHZ circuit (idle-filled) ops:  {len(circuit_ghz_filled.operations)}")

In [ ]:
# ── Noise model ───────────────────────────────────────────────────────────────
noise_ghz = profile.to_noise_model(
    circuit_ghz_filled, mode="t2", representation="pauli_twirl",
)

# ── ManyShotRunner — heatmap + GIF (Stim, fast at 15q) ───────────────────────
result_ghz = ManyShotRunner().run(
    circuit_ghz_filled, n_shots=N_SHOTS, noise_config=noise_ghz, seed=42,
)
print(f"ManyShotRunner done  zero-error fraction: {result_ghz.zero_error_fraction:.4f}")

# ── Trajectory pass — try 15q, fall back to 13q if OOM ───────────────────────
# 15q density matrix ≈ 8 GB RAM; 13q ≈ 512 MB.
# ManyShotRunner result above (heatmap/GIF) is unaffected by the fallback.
n_ghz_purity = N_GHZ
try:
    rho_ghz, pur_ghz = run_purity_pass(
        circuit_ghz_filled, noise_ghz, N_GHZ, f"GHZ {N_GHZ}q"
    )
except (MemoryError, ValueError):
    n_ghz_purity = 13
    print(f"OOM on {N_GHZ}q purity pass — rebuilding at {n_ghz_purity}q.")
    circuit_ghz_small = build_ghz_circuit(n_ghz_purity)
    circuit_ghz_small_filled = fill_idle_with_identities(circuit_ghz_small, gate_times)
    noise_ghz_small = profile.to_noise_model(
        circuit_ghz_small_filled, mode="t2", representation="pauli_twirl",
    )
    rho_ghz, pur_ghz = run_purity_pass(
        circuit_ghz_small_filled, noise_ghz_small, n_ghz_purity,
        f"GHZ {n_ghz_purity}q (OOM fallback)"
    )

In [ ]:
# Heatmap and GIF use the full 15q circuit.
# If the purity pass fell back to 13q, n_ghz_purity tracks that so the
# purity panel shows the correct number of qubit bars.
visualize_circuit(
    circuit_ghz_filled, result_ghz, noise_ghz, rho_ghz, pur_ghz,
    label=f"GHZ ({N_GHZ}q)",
    gif_name="ghz_15q",
    n_qubits_purity=n_ghz_purity,
)